In [68]:
import pandas as pd
import numpy as np

In [69]:
df = pd.read_csv('data/02_telco_cleaned.csv')

# **Data Transformation** 

## 1. **Feature Selection**

<small> The objective here is to take our cleaned columns and add, drop or construct new features that make it easier for a machine learning algorithm to spot patterns (like high churn risk).

##### **1.a) Dropping the Gender Column**

In [70]:
#Confirming the number of columns and rows in the dataframe
#The number of rows was 7,032 and columns was 20 after the data cleaning process.
print(df.shape)
print("The number of rows is: ", df.shape[0])
print("The number of columns is: ", df.shape[1])

(7032, 20)
The number of rows is:  7032
The number of columns is:  20


In [71]:
#Here we can check the columns of the dataframe to see what we have in it.
print(df.columns)

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')


<small> 

In the **Exploratory Data Analysis** stage, we discovered that the churn behaviour is nearly identical regardless of **'gender'**. The churn rates were as follows:

**Female customers:** ~26.9% churn rate

**Male customers:** ~26.2% churn rate

(Please refer to the chart 05_categorical_feature_analysis.png for visual representation.)

Therefore, this feature does not hold any predictive power on a machine learning model. I am going to drop it as this will simplify my data and reduce noise ensuring that the model is not picking up on random fluctuations.

In [72]:
# Dropping the 'gender' column as it is not needed for the analysis
df.drop('gender', axis=1, inplace=True)

#Verifying if the column has been dropped
print("New number of columns:", len(df.columns)) 

New number of columns: 19


##### **1.b) Dropping the TotalCharges Column**

<small> At the EDA stage, under the section 2.b) Numerical Analysis, we discovered a multicollinearity relationship between tenure and TotalCharges. 

**Correlation with tenure:** 0.83 (Extremely high)

**Correlation with MonthlyCharges:** 0.65 (High)

Linear models (like Logistic Regression) struggle with multicollineatry relationships because it makes it difficult for the model to estimate the independent effect of each variable.

**TotalCharges is essentially the mathematical product of tenure (months customer has been with the compnay) multiplied by MonthlyCharges.**


So the column TotalCharges will be dropped also to reduce noise in the model. However, tenure and MonthtlyCharges will be kept.



In [73]:
df.drop('TotalCharges', axis=1, inplace=True)

print("New number of columns is now:", len(df.columns)) 

New number of columns is now: 18


## 2. **Categorical Encoding**

Because machine learning frameworks operate purely on numerical data, an encoding pipeline was implemented to translate qualitative text categories into quantitative values.

In [75]:
#Here is a preview of the data types of each column in the dataframe. 
# This will help us understand what kind of data we are working with.
# We then can transform the data for modeling purposes.
df.dtypes

SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
Churn                   str
dtype: object

<small>The only numerical columns are SeniorCitizen, tenure and MonthlyCharges. The rest are text values, and will be transfomed into numbers in following steps. 

### **2.a) Target Variable Encoding**

<small>The target variable is **Churn** and is represented by the text values 'Yes' or 'No'. These must be converted to '1' or '0's inorder to comply with the machine learning frameworks.

In [76]:
# Manual clean mapping for binary columns
numerical_mappings = {
    'Churn': {'Yes': 1, 'No': 0},
    'Partner': {'No': 0, 'Yes': 1},
    'Dependents': {'No': 0, 'Yes': 1},
    'PhoneService': {'No': 0, 'Yes': 1},
    'PaperlessBilling': {'No': 0, 'Yes': 1},
    'Contract': {'Month-to-month': 0, 'One year': 1, 'Two year': 2},
}

for col, mapping in numerical_mappings.items():
    df[col] = df[col].map(mapping)

In [77]:
#One hot encoding for categorical columns
one_hot_columns = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'PaymentMethod', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV','StreamingMovies']

df = pd.get_dummies(df, columns=one_hot_columns, drop_first=True)

In [78]:
#Convert all True/False columns to 1/0 automatically
df = df.astype({col: int for col in df.select_dtypes(include='bool').columns})

In [79]:
df.dtypes

SeniorCitizen                              int64
Partner                                    int64
Dependents                                 int64
tenure                                     int64
PhoneService                               int64
Contract                                   int64
PaperlessBilling                           int64
MonthlyCharges                           float64
Churn                                      int64
MultipleLines_No phone service             int64
MultipleLines_Yes                          int64
InternetService_Fiber optic                int64
InternetService_No                         int64
OnlineSecurity_No internet service         int64
OnlineSecurity_Yes                         int64
PaymentMethod_Credit card (automatic)      int64
PaymentMethod_Electronic check             int64
PaymentMethod_Mailed check                 int64
OnlineBackup_No internet service           int64
OnlineBackup_Yes                           int64
DeviceProtection_No 

In [80]:
df.to_csv("data/03_telco_transformed.csv", index=False)